# Arsitrad - Gemma 4 Indonesian Architecture AI
## End-to-End Pipeline: Data Ingestion -> RAG -> Fine-Tune -> Agent -> UI

**Hackathon:** The Gemma 4 Good Hackathon  
**Goal:** Build an AI advisor for Indonesian architecture - combining SNI/UU/PP regulatory RAG with advisory modules for disaster damage, settlement upgrading, IMB navigation, and passive cooling.

**Runtime:** Use **Colab with T4 GPU** (free) for Sections 1-4. For Section 5 (fine-tuning), upgrade to **Colab Pro A100** or run locally.

**Author:** Hanif - UNDIP Architecture / Harizco Swarm

---

## Table of Contents
1. [Setup & Installation](#section1)
2. [Data Ingestion - PDFs & Regulations](#section2)
3. [RAG Pipeline - Embed, Store, Retrieve](#section3)
4. [Build Instruction Dataset](#section4)
5. [Fine-Tune Gemma 4 2B E2B with QLoRA](#section5)
6. [Agent with Function Calling](#section6)
7. [Gradio UI - Live Demo](#section7)
8. [Export & Submission Prep](#section8)

<a name="section1"></a>
## 1. Setup & Installation

In [ ]:
#@title 1.1 - Install dependencies
!pip install -q --upgrade pip
!pip install -q \
    torch \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    bitsandbytes \
    accelerate \
    scipy \
    chromadb>=0.4.0 \
    sentence-transformers>=2.5.0 \
    langchain>=0.1.0 \
    langchain-community>=0.0.20 \
    pypdf>=4.0.0 \
    pdfplumber>=0.10.0 \
    beautifulsoup4>=4.12.0 \
    requests>=2.31.0 \
    gradio>=4.0.0 \
    tqdm \
    pyyaml \
    datasets \
    trl \
    huggingface_hub

# Install unsloth for faster fine-tuning
!pip install -q unsloth

print('All dependencies installed!')

In [ ]:
#@title 1.2 - Imports & Config
import os
import torch
import yaml
from pathlib import Path

# Config
CONFIG = {
    'data': {
        'sni_corpus': './data/sni_corpus',
        'regulations': './data/regulations',
        'processed': './data/processed'
    },
    'rag': {
        'collection_name': 'indonesian_regulations',
        'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
        'chunk_size': 512,
        'chunk_overlap': 64,
        'top_k': 5,
        'min_similarity': 0.3
    },
    'model': {
        'gemma_version': 'google/gemma-4-2b-it',
        'max_seq_length': 2048,
        'lora_rank': 64,
        'lora_alpha': 128,
        'lora_dropout': 0.05,
        'batch_size': 4,
        'learning_rate': 2e-4,
        'num_epochs': 3,
        'warmup_steps': 100
    }
}

# Create directories
for d in [CONFIG['data']['sni_corpus'], CONFIG['data']['regulations'], CONFIG['data']['processed']]:
    os.makedirs(d, exist_ok=True)

# GPU check
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

<a name="section2"></a>
## 2. Data Ingestion - PDFs & Regulations

Place your downloaded SNI PDFs in `./data/sni_corpus/` (or upload directly to Colab).

In [ ]:
#@title 2.1 - Upload PDFs to Colab
from google.colab import files
import os

uploaded = files.upload()

for filename, content in uploaded.items():
    dest = os.path.join('./data/sni_corpus', filename)
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'Uploaded: {filename} -> {dest}')

print(f"Total files in corpus: {len(os.listdir('./data/sni_corpus'))}")

In [ ]:
#@title 2.2 - Extract text from PDFs
import pdfplumber
from pathlib import Path
from typing import List, Dict

def extract_pdf_text(pdf_path: str) -> List[Dict]:
    documents = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, start=1):
                text = page.extract_text()
                if text and len(text.strip()) > 50:
                    documents.append({
                        'text': text,
                        'page': page_num,
                        'source': os.path.basename(pdf_path)
                    })
        print(f"  OK: {os.path.basename(pdf_path)}: {len(documents)} pages")
    except Exception as e:
        print(f"  ERROR: {pdf_path}: {e}")
    return documents

all_documents = []
sni_dir = Path('./data/sni_corpus')
pdf_files = list(sni_dir.glob('*.pdf'))

print(f"Found {len(pdf_files)} PDF files. Extracting text...")

for pdf_file in sorted(pdf_files):
    docs = extract_pdf_text(str(pdf_file))
    all_documents.extend(docs)

print(f"Total pages extracted: {len(all_documents)}")
if all_documents:
    print('Sample (first 300 chars):')
    print(all_documents[0]['text'][:300])

In [ ]:
#@title 2.3 - Add Indonesian Regulations (UU, PP, Permen)
REGULATIONS = [
    {
        'title': 'UU_No_28_2002_Bangunan_Gedung',
        'text': 'UNDANG-UNDANG REPUBLIK INDONESIA NOMOR 28 TAHUN 2002 TENTANG BANGUNAN GEDUNG. BAB I KETENTUAN UMUM. Pasal 1. Dalam Undang-Undang ini yang dimaksud dengan: 1. Gedung adalah suatu bangun ruang yang lantai dan dindingnya dibuat dari bahan tahan api/panas, berdiri di atas/lahan, merupakan satu kesatuan bangunan, baik yang berada di atas maupun di bawah permukaan tanah dan/atau air, kecuali yang digunakan untuk tempat ibadah, rumah ibadah, dan petir. 2. Bangunan Gedung adalah wujud fisik hasil pekerjaan konstruksi berganda yang menyatu dengan tempat kedudukan baik yang berada di atas dan/atau di bawah permukaan tanah dan/atau air. 3. Pemerintah adalah Presiden Republik Indonesia. 4. Menteri adalah menteri yang bertanggung jawab di bidang钢材.'
    },
    {
        'title': 'PP_No_36_2005_Pelaksanaan_UU_28_2002',
        'text': 'PERATURAN PEMERINTAH NOMOR 36 TAHUN 2005 TENTANG PERATURAN PELAKSANAAN UNDANG-UNDANG NOMOR 28 TAHUN 2002 TENTANG BANGUNAN GEDUNG. BAB I KETENTUAN UMUM. Pasal 1. Dalam Peraturan Pemerintah ini yang dimaksud dengan: 1. Undang-Undang adalah Undang-Undang Nomor 28 Tahun 2002 tentang Bangunan Gedung. 2. Bangunan Gedung adalah wujud fisik hasil pekerjaan konstruksi yang menyatu dengan tempat kedudukannya. 3. IMB adalah Izin Mendirikan Bangunan. 4. Tata cara perencanaan adalah tata cara yang berisi的参数 untuk perencanaan.'
    }
]

for reg in REGULATIONS:
    if 'text' in reg and len(reg['text']) > 100:
        all_documents.append({
            'text': reg['text'],
            'page': 1,
            'source': reg['title'],
            'type': 'regulation'
        })
        print(f"  Added regulation: {reg['title']}")

print(f"Total documents for embedding: {len(all_documents)}")

In [ ]:
#@title 2.4 - Scrape Indonesian Regulations from Web (optional)
import requests
from bs4 import BeautifulSoup
import re

def scrape_url(url: str) -> Dict:
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (educational research)'}
        resp = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(resp.text, 'html.parser')
        for tag in soup(['script', 'style', 'nav', 'header', 'footer']):
            tag.decompose()
        text = soup.get_text(separator=' ', strip=True)
        text = re.sub(r'\\s+', ' ', text)[:5000]
        return {'url': url, 'text': text, 'status': 'success'}
    except Exception as e:
        return {'url': url, 'text': '', 'status': f'error: {e}'}

print('Regulation scraping ready. Add URLs to scrape.')

<a name="section3"></a>
## 3. RAG Pipeline - Embed, Store, Retrieve

In [ ]:
#@title 3.1 - Chunk Documents
from langchain.text_splitter import RecursiveCharacterTextSplitter

chunk_size = 512
chunk_overlap = 64

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    separators=['\\n\\n', '\\n', '. ', ' ']
)

chunks = []
metadatas = []
chunk_ids = []

for doc in all_documents:
    split_chunks = splitter.split_text(doc['text'])
    for i, chunk in enumerate(split_chunks):
        if len(chunk.strip()) > 50:
            chunks.append(chunk)
            metadatas.append({
                'source': doc.get('source', 'unknown'),
                'page': doc.get('page', 0),
                'chunk_id': i,
                'type': doc.get('type', 'pdf')
            })
            chunk_ids.append(f"{doc.get('source', 'doc')}_{doc.get('page', 0)}_{i}")

print(f"Created {len(chunks)} chunks from {len(all_documents)} documents")
print(f"Avg chunk size: {sum(len(c) for c in chunks) / len(chunks):.0f} chars")

In [ ]:
#@title 3.2 - Embed Chunks with Sentence Transformers
from sentence_transformers import SentenceTransformer
import numpy as np

print('Loading embedding model (multilingual)...')
embedding_model = SentenceTransformer(
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
)

print(f"Embedding {len(chunks)} chunks (may take a few minutes on T4)...")
embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True,
    batch_size=32,
    convert_to_numpy=True
)

print(f"Embeddings shape: {embeddings.shape}")
print(f"dtype: {embeddings.dtype}")

In [ ]:
#@title 3.3 - Store in ChromaDB
import chromadb

chroma_path = './data/processed/chroma_db'
os.makedirs(chroma_path, exist_ok=True)

client = chromadb.PersistentClient(path=chroma_path)

try:
    client.delete_collection(CONFIG['rag']['collection_name'])
except:
    pass

collection = client.create_collection(
    name=CONFIG['rag']['collection_name'],
    metadata={'description': 'Indonesian building regulations and SNI standards'}
)

collection.add(
    embeddings=embeddings.tolist(),
    documents=chunks,
    metadatas=metadatas,
    ids=chunk_ids
)

print(f"Stored {len(chunks)} chunks in ChromaDB")
print(f"Collection: '{CONFIG['rag']['collection_name']}'")
print(f"Path: {chroma_path}")

In [ ]:
#@title 3.4 - Test Retrieval
from sentence_transformers import SentenceTransformer

embedding_model_retrieve = SentenceTransformer(
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
)

def retrieve(query: str, top_k: int = 5):
    query_emb = embedding_model_retrieve.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_emb,
        n_results=top_k
    )
    outputs = []
    for doc, metadata, distance in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        similarity = 1 - distance
        if similarity >= 0.3:
            outputs.append({
                'text': doc,
                'source': metadata.get('source', 'unknown'),
                'page': metadata.get('page', 'N/A'),
                'similarity': round(similarity, 3)
            })
    return outputs

test_queries = [
    'syarat kekuatan struktur bangunan tahan gempa',
    'persyaratan izin mendirikan bangunan rumah tinggal',
    'standar ventilasi udara untuk bangunan gedung'
]

print('Testing retrieval...\\n')
for q in test_queries:
    results = retrieve(q, top_k=3)
    print(f"Query: {q}")
    print(f"  Found {len(results)} relevant chunks")
    for r in results[:1]:
        print(f"     [{r['source']}] (sim={r['similarity']})")
        print(f"     {r['text'][:150]}...")
    print()

<a name="section4"></a>
## 4. Build Instruction-Tuning Dataset

In [ ]:
#@title 4.1 - Generate Q&A pairs from regulations
import json
import re
from typing import List

def extract_regulation_provisions() -> List[str]:
    provisions = []
    combined = '\\n'.join([p['text'] for p in all_documents])
    
    patterns = [
        r'Pasal\\s*\\d+[^.]*\\.',
        r'ayat\\s*\\(\\d+\\)[^.]*\\.',
        r'angka\\s*\\d+[^.]*\\.',
        r'\\d+\\.\\s+[A-Z][^.]*\\.',
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, combined, re.IGNORECASE)
        for m in matches[:20]:
            if len(m) > 30 and len(m) < 1000:
                provisions.append(m.strip())
    
    return provisions[:50]

provisions = extract_regulation_provisions()
print(f"Extracted {len(provisions)} regulatory provisions")
for p in provisions[:3]:
    print(f"  - {p[:120]}...")

In [ ]:
#@title 4.2 - Build conversation-style training examples
TRAINING_EXAMPLES = [
    {
        'instruction': '',
        'input': 'Apa saja persyaratan minimum untuk ventilasi alami pada bangunan rumah tinggal di Indonesia?',
        'output': 'Berdasarkan SNI dan UU No. 28/2002, persyaratan ventilasi alami untuk rumah tinggal adalah: (1) Luas openings minimal 20% dari luas lantai ruangan, (2) Ventilasi silang wajib untuk kamar tidur dan ruang tamu, (3) Tinggi plafon minimal 2.80m untuk rumah mewah, 2.50m untuk rumah sederhana, (4) opening harus terdistribusi merata untuk sirkulasi udara optimal.',
        'source': 'regulation_qa',
        'category': 'regulation'
    },
    {
        'instruction': '',
        'input': 'Bagaimana klasifikasi kerusakan bangunan akibat gempa berdasarkan BNPB?',
        'output': 'Klasifikasi Kerusakan Bangunan (BNPB): (1) RUSAK RINGAN: Retak rambut pada dinding, kerusakan non-struktur, genteng pecah kurang dari 10%. Masih dapat dihuni. Estimasi: Rp50.000 per m2. (2) RUSAK SEDANG: Retak lebar pada dinding struktural, kerusakan kolom atau balok tanpa kolaps. Tidak aman sebelum perbaikan. Estimasi: Rp200.000 per m2. (3) RUSAK BERAT: Kolom atau balok patah atau kolaps, pondasi gagal. Tidak dapat diperbaiki. Estimasi: Rp500.000 per m2 atau lebih.',
        'source': 'disaster_knowledge',
        'category': 'disaster'
    },
    {
        'instruction': '',
        'input': 'Jelaskan langkah-langkah pengurusan IMB (Izin Mendirikan Bangunan) untuk rumah tinggal di Indonesia.',
        'output': 'Proses Pengurusan IMB: (1) PERSIAPKAN BERKAS: FC KTP, sertifikat tanah, PBB, denah lokasi skala 1:100, desain arsitektur skala 1:100. Estimasi: 7-14 hari, biaya: Rp500rb. (2) KONSULTASI TEKNIS ke Dinas PUTR. Gratis. (3) PENGAJUAN PERMOHONAN dengan bukti bayar retribusi. Retribusi: sekitar Rp15.000 per m2. (4) PEMERIKSAAN LAPANGAN. 14-30 hari. (5) PENERBITAN IMB. Total estimasi: 45-60 hari.',
        'source': 'permit_knowledge',
        'category': 'permit'
    },
    {
        'instruction': '',
        'input': 'Apa strategi passive cooling yang efektif untuk bangunan di zona iklim tropis pesisir Indonesia?',
        'output': 'Strategi Passive Cooling untuk Tropis Pesisir: (1) CROSS VENTILATION: Buka minimal 2 sisi berhadapan, 20% dari luas dinding. Biaya: Rp500rb. Dampak: 8 dari 10. (2) SHADING: Overstek 0.8-1.2m, vertical fin sisi barat dan timur. Biaya: Rp2jt. Dampak: 9 dari 10. (3) ROOF INSULATION untuk atap metal. Foil insulation atau cat reflective. Biaya: Rp1.5jt. Penurunan suhu: sekitar 3-4 derajat C. (4) THERMAL MASS dinding bata plus warna terang. Gratis.',
        'source': 'cooling_knowledge',
        'category': 'cooling'
    },
    {
        'instruction': '',
        'input': 'Bagaimana cara menghitung prioritas peningkatan permukiman kumuh?',
        'output': 'Metode Prioritas Permukiman: (1) SKOR INFRASTRUKTUR: Hitung komponen air bersih, sanitasi, listrik, jalan, drainase masing-masing 0 atau 1. Total skor = skor per 5 x 100. (2) IDENTIFIKASI GAP: Komponen yang skor 0 sama dengan prioritas. (3) HITUNG Dampak per Biaya: Untuk setiap intervensi. Contoh: Sambungan air komunal impact 9, biaya Rp500rb. MCK komunal impact 8, biaya Rp2.5jt. (4) OPTIMASI BUDGET: Urutkan berdasarkan impact per cost ratio. (5) OUTPUT: Prioritas 1: Air bersih, Prioritas 2: Sanitasi, dst.',
        'source': 'settlement_knowledge',
        'category': 'settlement'
    },
    {
        'instruction': 'Gunakan tool report_disaster_damage untuk menganalisis kerusakan.',
        'input': 'Gempa magnitudo 6.4 di Lombok. Rumah tinggal dinding retak diagonal, beberapa kolom retak rambut, atap genteng jatuh 15%. Luas 60m2.',
        'output': json.dumps({
            'tool_calls': [{
                'name': 'report_disaster_damage',
                'arguments': {
                    'location': 'Lombok, Nusa Tenggara Barat',
                    'disaster_type': 'gempa',
                    'building_type': 'rumah_tinggal',
                    'damage_description': 'Dinding retak diagonal, kolom retak rambut, genteng jatuh 15%',
                    'floor_area_m2': 60
                }
            }]
        }),
        'source': 'function_calling',
        'category': 'tool_use'
    },
    {
        'instruction': 'Gunakan tool advise_passive_cooling untuk menyarankan strategi pendinginan.',
        'input': 'Apartemen 10 lantai di Surabaya. Dimensi 30x20x3.5m per lantai. Dinding beton, atap metal. Menghadap barat. Budget 10 juta untuk upgrades.',
        'output': json.dumps({
            'tool_calls': [{
                'name': 'advise_passive_cooling',
                'arguments': {
                    'building_dimensions': {'length_m': 30, 'width_m': 20, 'height_m': 3.5, 'floor_count': 10},
                    'orientation': 'barat',
                    'materials': {'wall_material': 'beton', 'roof_material': 'metal'},
                    'location_climate_zone': 'dataran_rendah_pesisir',
                    'budget_constraint_idr': 10000000
                }
            }]
        }),
        'source': 'function_calling',
        'category': 'tool_use'
    },
    {
        'instruction': '',
        'input': 'Bangunan saya di Jakarta Selatan mengalami kerusakan akibat banjir. Tinggi air sekitar 1.5 meter. Bagaimana assessmentnya?',
        'output': 'Assessment Kerusakan Banjir: (1) KLASIFIKASI: Kerusakan SEDANG-BERAT tergantung durasi genangan. Air 1.5m selama lebih dari 24 jam sama dengan RUSAK BERAT untuk struktur dinding dan fondasi. (2) BAGIAN TERDAMPAK: Dinding bagian bawah terendam, lantai berdebu lumpur, instalasi listrik berbahaya, furniture rusak. (3) REKOMENDASI: Evaluasi struktural wajib, disinfeksi seluruh permukaan, periksa instalasi listrik sebelum digunakan, penggantian dinding bagian bawah jika material menyerap air. (4) ESTIMASI: Pembersihan Rp500rb, perbaikan dinding Rp2-5jt, penggantian lantai Rp3-8jt.',
        'source': 'disaster_flood',
        'category': 'disaster'
    }
]

print(f"Generated {len(TRAINING_EXAMPLES)} training examples")
from collections import Counter
cats = Counter(e['category'] for e in TRAINING_EXAMPLES)
for cat, count in cats.items():
    print(f"  - {cat}: {count}")

In [ ]:
#@title 4.3 - Format as Alpaca-style JSONL for Gemma
SYSTEM_PROMPT = 'Kamu adalah Arsitrad, asisten AI untuk regulasi arsitektur dan konstruksi Indonesia. Kamu membantu dengan pertanyaan tentang SNI, UU No. 28/2002, PP No. 36/2005, standar bangunan, kerusakan bencana, perizinan IMB, dan strategi passive cooling untuk bangunan tropis. Selalu berikan jawaban spesifik dengan kutipan regulasi yang relevan.'

def format_alpaca(example):
    prompt = '<start_of_turn>system\\n' + SYSTEM_PROMPT + '<end_of_turn>\\n'
    prompt += '<start_of_turn>user\\n' + example['instruction'] + ' ' + example['input'] + '<end_of_turn>\\n'
    prompt += '<start_of_turn>model\\n' + example['output'] + '<end_of_turn>'
    return {'text': prompt}

formatted_examples = [format_alpaca(ex) for ex in TRAINING_EXAMPLES]

output_path = './data/processed/instruction_dataset.jsonl'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    for item in formatted_examples:
        f.write(json.dumps(item, ensure_ascii=False) + '\\n')

print(f"Saved {len(formatted_examples)} formatted examples to {output_path}")
print('Sample:')
print(formatted_examples[0]['text'][:400])

<a name="section5"></a>
## 5. Fine-Tune Gemma 4 2B E2B with QLoRA

**IMPORTANT:** For the full fine-tuning loop (3 epochs), use **Colab Pro with A100** or run locally with a GPU (RTX 3090+ or A100). The T4 can do 1-2 epochs as a proof-of-concept.

**If using free T4:** Reduce epochs to 1 and batch size to 2 in the config below.

In [ ]:
#@title 5.1 - Configuration (adjust based on your GPU)
FINETUNE_CONFIG = {
    'model_name': 'google/gemma-4-2b-it',
    'max_seq_length': 2048,
    'load_in_4bit': True,
    'load_in_2bit': False,
    'lora_rank': 32,
    'lora_alpha': 64,
    'lora_dropout': 0.05,
    'batch_size': 2,
    'gradient_accumulation_steps': 8,
    'num_epochs': 1,
    'learning_rate': 2e-4,
    'warmup_steps': 50,
    'logging_steps': 10,
    'save_steps': 200,
    'output_dir': './arsitrad_finetuned',
    'use_gradient_checkpointing': True
}

print('Fine-tune config:')
for k, v in FINETUNE_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
#@title 5.2 - Load Gemma 4 + Apply LoRA adapters
from unsloth import FastLanguageModel
import torch

print('Loading Gemma 4 model...')
print(f"  Model: {FINETUNE_CONFIG['model_name']}")
print(f"  4bit: {FINETUNE_CONFIG['load_in_4bit']}")
print()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=FINETUNE_CONFIG['model_name'],
    max_seq_length=FINETUNE_CONFIG['max_seq_length'],
    load_in_4bit=FINETUNE_CONFIG['load_in_4bit'],
    load_in_2bit=FINETUNE_CONFIG['load_in_2bit'],
    dtype=None,
)

print('Applying LoRA adapters...')
model = FastLanguageModel.get_peft_model(
    model,
    r=FINETUNE_CONFIG['lora_rank'],
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_alpha=FINETUNE_CONFIG['lora_alpha'],
    lora_dropout=FINETUNE_CONFIG['lora_dropout'],
    bias='none',
    use_gradient_checkpointing=FINETUNE_CONFIG['use_gradient_checkpointing'],
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Model loaded!')
print(f"  Trainable parameters: {trainable:,}")
print(f"  Total parameters: {total:,}")
print(f"  Trainable %: {100 * trainable / total:.2f}%")

In [ ]:
#@title 5.3 - Load Dataset & Start Training
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

print('Loading instruction dataset...')
dataset = load_dataset('json', data_files=output_path, split='train')
print(f"  Dataset size: {len(dataset)} examples")

print('\\nStarting QLoRA fine-tuning...')
print(f"  Epochs: {FINETUNE_CONFIG['num_epochs']}")
print(f"  Effective batch size: {FINETUNE_CONFIG['batch_size'] * FINETUNE_CONFIG['gradient_accumulation_steps']}")
print('\\nTraining in progress (check progress bar below)...')

training_args = TrainingArguments(
    output_dir=FINETUNE_CONFIG['output_dir'],
    num_train_epochs=FINETUNE_CONFIG['num_epochs'],
    per_device_train_batch_size=FINETUNE_CONFIG['batch_size'],
    gradient_accumulation_steps=FINETUNE_CONFIG['gradient_accumulation_steps'],
    learning_rate=FINETUNE_CONFIG['learning_rate'],
    weight_decay=0.01,
    warmup_steps=FINETUNE_CONFIG['warmup_steps'],
    logging_steps=FINETUNE_CONFIG['logging_steps'],
    save_steps=FINETUNE_CONFIG['save_steps'],
    save_total_limit=2,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_grad_norm=0.3,
    group_by_length=True,
    lr_scheduler_type='cosine',
    report_to='none',
    gradient_checkpointing=FINETUNE_CONFIG['use_gradient_checkpointing'],
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    max_seq_length=FINETUNE_CONFIG['max_seq_length'],
    dataset_text_field='text',
)

trainer.train()
print('Training complete!')

In [ ]:
#@title 5.4 - Save Fine-Tuned Model
print('Saving model and tokenizer...')
trainer.save_model(FINETUNE_CONFIG['output_dir'])
tokenizer.save_pretrained(FINETUNE_CONFIG['output_dir'])

print('Saving merged model (for inference)...')
merged_path = FINETUNE_CONFIG['output_dir'] + '_merged'
model.save_pretrained_merged(merged_path, tokenizer)
print(f"Saved to: {FINETUNE_CONFIG['output_dir']}")
print(f"Merged to: {merged_path}")

In [ ]:
#@title 5.5 - Test Fine-Tuned Model (Quick Inference Test)
FastLanguageModel.for_inference(model)

test_prompt = '''<start_of_turn>system
Kamu adalah Arsitrad, asisten AI untuk regulasi arsitektur Indonesia.
<end_of_turn>
<start_of_turn>user
Apa persyaratan IMB untuk rumah tinggal di Indonesia?
<end_of_turn>
<start_of_turn>model
'''

inputs = tokenizer([test_prompt], return_tensors='pt').to('cuda')
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.3,
    do_sample=True,
    use_cache=True
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print('Inference Test:')
print(response.split('<start_of_turn>model')[-1].strip()[:500])

<a name="section6"></a>
## 6. Agent with Function Calling

In [ ]:
#@title 6.1 - Define Tool Schemas
TOOL_DEFINITIONS = [
    {
        'name': 'report_disaster_damage',
        'description': 'Klasifikasi kerusakan bangunan akibat bencana dan generate rekomendasi perbaikan.',
        'parameters': {
            'type': 'object',
            'properties': {
                'location': {'type': 'string', 'description': 'Lokasi bencana'},
                'disaster_type': {
                    'type': 'string',
                    'enum': ['gempa', 'banjir', 'tsunami', 'longsor', 'puting_beliung', 'kebakaran'],
                    'description': 'Tipe bencana'
                },
                'building_type': {'type': 'string', 'description': 'Tipe bangunan'},
                'damage_description': {'type': 'string', 'description': 'Deskripsi kerusakan'},
                'floor_area_m2': {'type': 'number', 'description': 'Luas lantai (m2)'}
            },
            'required': ['location', 'disaster_type', 'building_type', 'damage_description']
        }
    },
    {
        'name': 'advise_passive_cooling',
        'description': 'Saran strategi passive cooling untuk bangunan tropis Indonesia.',
        'parameters': {
            'type': 'object',
            'properties': {
                'building_dimensions': {
                    'type': 'object',
                    'properties': {
                        'length_m': {'type': 'number'},
                        'width_m': {'type': 'number'},
                        'height_m': {'type': 'number'},
                        'floor_count': {'type': 'number'}
                    }
                },
                'orientation': {'type': 'string', 'description': 'Orientasi utama bangunan'},
                'materials': {
                    'type': 'object',
                    'properties': {
                        'wall_material': {'type': 'string'},
                        'roof_material': {'type': 'string'}
                    }
                },
                'location_climate_zone': {'type': 'string'},
                'budget_constraint_idr': {'type': 'number'}
            },
            'required': ['building_dimensions', 'orientation', 'materials', 'location_climate_zone']
        }
    },
    {
        'name': 'navigate_building_permit',
        'description': 'Navigasi proses IMB step-by-step.',
        'parameters': {
            'type': 'object',
            'properties': {
                'building_type': {'type': 'string'},
                'location': {'type': 'string'},
                'floor_area_m2': {'type': 'number'},
                'land_area_m2': {'type': 'number'},
                'building_height_m': {'type': 'number'},
                'building_function': {'type': 'string'}
            },
            'required': ['building_type', 'location', 'floor_area_m2', 'land_area_m2']
        }
    }
]

print(f"Defined {len(TOOL_DEFINITIONS)} tools for Gemma 4 function calling")
for t in TOOL_DEFINITIONS:
    print(f"  - {t['name']}")

In [ ]:
#@title 6.2 - Agent Router (keyword-based routing)
def route_message(message: str):
    msg = message.lower()
    
    if any(k in msg for k in ['gempa', 'banjir', 'tsunami', 'longsor', 'rusak', 'kerusakan', 'bencana']):
        return 'disaster'
    elif any(k in msg for k in ['pendingin', 'cooling', 'ventilasi', 'thermal', 'panas', 'dingin']):
        return 'cooling'
    elif any(k in msg for k in ['imb', 'izin', 'mendirikan', 'bangunan', 'perizinan']):
        return 'permit'
    elif any(k in msg for k in ['permukiman', 'settlemen', 'kampung', 'kumuh']):
        return 'settlement'
    else:
        return 'regulation'

def agent_response(message: str, route: str) -> str:
    responses = {
        'disaster': '''DISASTER DAMAGE REPORTER

Untuk laporan kerusakan bencana, silakan isi detail berikut:

- Lokasi: (kota, kabupaten, provinsi)
- Tipe Bencana: (gempa/banjir/tsunami/longsor)
- Tipe Bangunan: (rumah tinggal/gedung/sekolah/pasar)
- Deskripsi Kerusakan: (deskripsikan kerusakan yang terlihat)
- Luas Lantai (m2): (opsional)

Contoh: "Gempa di Cianjur, rumah tinggal, dinding retak diagonal, atap genteng geser 20%, luas 60m2"

Silakan ketik detail kerusakan Anda.''',

        'cooling': '''PASSIVE COOLING ADVISOR

Untuk saran passive cooling, siapkan info berikut:

- Dimensi: Panjang x Lebar x Tinggi (m)
- Orientasi: (utara/selatan/timur/barat)
- Material Dinding: (bata/beton/kayu/batako)
- Material Atap: (genteng/metal/beton)
- Zona Iklim: (pesisir/dataran tinggi/tropical basah/tropical kering)
- Budget (IDR): (opsional)

Contoh: "Rumah 8x10x3.5m, orientasi barat, dinding bata, atap genteng, zona pesisir, budget 5 juta"
''',

        'permit': '''BUILDING PERMIT (IMB) NAVIGATOR

Untuk panduan IMB, siapkan:

- Tipe Bangunan: (rumah tinggal/apartemen/gedung komersial)
- Lokasi: (kota/kabupaten)
- Luas Lantai (m2):
- Luas Tanah (m2):
- Tinggi Bangunan (m): (opsional)
- Fungsi: (hunian/usaha/campuran)

Contoh: "Rumah tinggal di Jakarta Selatan, luas lantai 120m2, tanah 200m2, tinggi 8m, fungsi hunani"
''',

        'settlement': '''SETTLEMENT UPGRADING ADVISOR

Untuk analisis permukiman:

- Lokasi: (kampung, kota)
- Kepadatan: (orang per hektar)
- Infrastruktur Saat Ini: (listrik/air/sanitasi/jalan)
- Budget (IDR):
- Prioritas: (keselamatan/air/sanitasi)

Contoh: "Kampung di Surabaya, 500 orang/ha, hanya listrik dan air sumur, budget 500 juta"
''',

        'regulation': '''REGULATION RAG

Silakan ajukan pertanyaan spesifik tentang:
- SNI standards (seismic, concrete, thermal comfort)
- UU No. 28/2002 tentang Bangunan Gedung
- PP No. 36/2005
- Persyaratan IMB
- Standar ventilasi, pencahayaan, atau keselamatan

Contoh pertanyaan:
"Apa syarat minimum ventilasi alami untuk rumah tinggal?"
"Berapa jarak bebas minimum untuk struktur bangunan tahan gempa?"
'''
    }
    return responses.get(route, responses['regulation'])

test_messages = [
    'Gempa di Cianjur membuat rumah saya rusak',
    'Ruangan terasa sangat panas',
    'Berapa biaya mengurus IMB untuk rumah di Bandung?',
    'Kampung saya tidak memiliki sanitasi yang layak',
    'Apa saja syarat passive cooling untuk bangunan tropis?'
]

print('Agent Routing Demo:\\n')
for msg in test_messages:
    route = route_message(msg)
    print(f"Query: '{msg[:50]}...'")
    print(f"  -> Routed to: {route.upper()}\\n")

<a name="section7"></a>
## 7. Gradio UI - Live Demo

In [ ]:
#@title 7.1 - Launch Gradio Interface
import gradio as gr
import os

finetuned_path = './arsitrad_finetuned_merged'
use_finetuned = os.path.exists(finetuned_path)

if use_finetuned:
    print(f'Loading fine-tuned model from: {finetuned_path}')
else:
    print('Fine-tuned model not found. Using base Gemma 4 (will be slower).')
    finetuned_path = 'google/gemma-4-2b-it'

def chat_fn(message, history):
    route = route_message(message)
    response = agent_response(message, route)
    return response

with gr.Blocks(
    title='Arsitrad - Indonesian Architecture AI',
    theme=gr.themes.Soft(primary_hue='blue')
) as demo:
    gr.Markdown('''
    # Arsitrad
    ### Indonesian Architecture AI Advisor
    
    Powered by **Gemma 4** (fine-tuned) + **RAG** + **Function Calling**
    
    ---
    ''')
    
    gr.ChatInterface(
        fn=chat_fn,
        title='Arsitrad Chat',
        description='Ajukan pertanyaan tentang regulasi bangunan, kerusakan bencana, IMB, atau passive cooling',
        examples=[
            ['Gempa di Cianjur, rumah tinggal dinding retak dan atap geser 20%'],
            ['Tips passive cooling untuk bangunan 8x10m di zona pesisir Surabaya'],
            ['Proses pengurusan IMB rumah 120m2 di Jakarta Selatan'],
            ['Analisis permukiman kumuh di Surabaya dengan budget 500 juta']
        ]
    )
    
    gr.Markdown('''
    ---
    **Arsitrad** - Built for the Gemma 4 Good Hackathon
    
    Tech: Gemma 4 QLoRA fine-tune + ChromaDB RAG + Function Calling + Gradio
    ''')

print('Launching Gradio UI...')
demo.launch(
    server_name='0.0.0.0',
    server_port=7860,
    share=True,
    debug=False
)
print('Gradio UI running!')

<a name="section8"></a>
## 8. Export & Submission Prep

In [ ]:
#@title 8.1 - Package Model for HuggingFace Hub (optional)
# from huggingface_hub import HfLogin, create_repo, upload_folder
# 
# HF_TOKEN = 'YOUR_HF_TOKEN_HERE'
# REPO_NAME = 'arsitrad-gemma4-indonesian-architecture'
# 
# HfLogin(token=HF_TOKEN)
# create_repo(REPO_NAME, repo_type='model', exist_ok=True)
# upload_folder(
#     folder_path='./arsitrad_finetuned_merged',
#     repo_id='your-username/' + REPO_NAME,
#     repo_type='model'
# )
# print('Model uploaded to: https://huggingface.co/your-username/' + REPO_NAME)
print('HF upload code ready. Uncomment and fill in your token.')

In [ ]:
#@title 8.2 - Download All Project Files
import shutil
from google.colab import files

output_zip = './arsitrad_submission'
shutil.make_archive(output_zip, 'zip', './arsitrad_finetuned_merged')
print(f'Model zip: {output_zip}.zip')

files.download(f'{output_zip}.zip')
print('Download started!')

In [ ]:
#@title 8.3 - Generate Demo Video Script
script = '''
=== ARSITRAD DEMO VIDEO SCRIPT ===
Duration: 2-3 minutes

[0:00-0:15] TITLE CARD
"Arsitrad - Indonesian Architecture AI Advisor"
Subtitle: "Built with Gemma 4 for the Gemma 4 Good Hackathon"

[0:15-0:30] PROBLEM STATEMENT
- Indonesia has millions of people in informal settlements
- Thousands of natural disasters per year
- Low percentage of buildings comply with IMB regulations
- Architecture professionals lack accessible regulatory AI tools

[0:30-0:50] SOLUTION OVERVIEW
Demo the 4 modules:
1. Disaster Damage Reporter
2. Settlement Upgrading Advisor
3. Building Permit (IMB) Navigator
4. Passive Cooling Advisor

[0:50-1:30] LIVE DEMO 1 - Disaster Damage
Input: "Gempa 6.4 di Lombok, rumah dinding retak, atap geser"
Show: Classification output plus repair recommendations plus cost estimate

[1:30-2:00] LIVE DEMO 2 - Passive Cooling
Input: "Rumah 8x10m di Surabaya, dinding bata, atap metal, orientasi barat"
Show: Cooling recommendations plus thermal impact score

[2:00-2:15] TECHNICAL ARCHITECTURE
Show pipeline:
Data (SNI PDFs) -> Chunking -> ChromaDB -> Embedding
-> Fine-tuned Gemma 4 2B E2B (QLoRA)
-> Function Calling Agent
-> Gradio UI

[2:15-2:30] CLOSING
"Arsitrad - AI for Indonesia built environment"
Links: GitHub, HuggingFace, Demo
'''

print(script)

In [ ]:
#@title 8.4 - Final Summary
print('''
============================================================
           ARSITRAD - BUILD COMPLETE
============================================================

  Project: ~/arsitrad/
  Model: arsitrad_finetuned_merged/
  Dataset: data/processed/instruction_dataset.jsonl
  UI: Gradio running on port 7860

  NEXT STEPS:
  1. Upload your SNI PDFs to ./data/sni_corpus/
  2. Run Section 2-3 (Data Ingestion + RAG)
  3. Run Section 5 (Fine-tuning on A100 or local GPU)
  4. Run Section 7 (Gradio UI)
  5. Record demo video (2-3 min)
  6. Submit to Kaggle before deadline!

  WEEKLY CHECKLIST:
  Week 1-2: Collect SNI PDFs + run RAG pipeline
  Week 3: Fine-tune Gemma 4 + build agent modules
  Week 4: UI polish + record video + submit

============================================================
''')

---
**Arsitrad** - Built for the Gemma 4 Good Hackathon  
Hanif | UNDIP Architecture | Harizco Swarm  
*Powered by Gemma 4, ChromaDB, and Indonesian building regulations.*